In [ ]:
# =========================================================
# IMPROVED DATASET PREPARATION
# AI Clinical Decision Support Assistant
# =========================================================

# =========================================================
# STEP 1 — IMPORT LIBRARIES
# =========================================================

import pandas as pd
import random

# =========================================================
# STEP 2 — LOAD DATASETS
# =========================================================

heart_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/heart.csv"
)

symptom_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/Healthcare.csv"
)

lifestyle_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/heart_2022_no_nans.csv"
)

knowledge_df = pd.read_excel(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/disease_knowledge.xlsx"
)

# =========================================================
# STEP 3 — CLEAN COLUMN NAMES
# =========================================================

heart_df.columns = heart_df.columns.str.strip()
symptom_df.columns = symptom_df.columns.str.strip()
lifestyle_df.columns = lifestyle_df.columns.str.strip()
knowledge_df.columns = knowledge_df.columns.str.strip()

# =========================================================
# STEP 4 — CREATE DISEASE KNOWLEDGE DICTIONARY
# =========================================================

disease_info = {}

for _, row in knowledge_df.iterrows():

    disease_info[row["disease"]] = {

        "tests": row["tests"],

        "recommendation": row["recommendation"],

        "precautions": row["precautions"]
    }

# =========================================================
# STEP 5 — ADD LOW RISK CASE
# =========================================================

disease_info["Low Cardiovascular Risk"] = {

    "tests": "Routine Health Checkup",

    "recommendation": "Maintain healthy lifestyle and regular exercise",

    "precautions": "Monitor symptoms if they worsen"
}

# =========================================================
# STEP 6 — CARDIAC SYMPTOM FILTER
# =========================================================

cardiac_keywords = [

    "chest pain",
    "shortness of breath",
    "fatigue",
    "dizziness",
    "palpitations",
    "irregular heartbeat",
    "sweating",
    "swelling",
    "left arm pain",
    "chest tightness",
    "fainting",
    "weakness"
]

# =========================================================
# STEP 7 — SECONDARY DISEASE MAPPING
# =========================================================

def detect_secondary_disease(primary):

    mapping = {

        "Heart Attack": "Coronary Artery Disease",

        "Arrhythmia": "Atrial Fibrillation",

        "Heart Failure": "Hypertension",

        "Angina": "Coronary Artery Disease",

        "Hypertension": "Coronary Artery Disease",

        "Coronary Artery Disease": "Angina",

        "Low Cardiovascular Risk": "Hypertension"
    }

    return mapping.get(primary, "Coronary Artery Disease")

# =========================================================
# STEP 8 — RULE-BASED DISEASE DETECTION
# =========================================================

def detect_disease(symptoms, bp, chol, heart_rate, diabetes, bmi):

    symptoms = symptoms.lower()

    # =====================================================
    # LOW RISK CASE
    # =====================================================

    if (

        bp < 130
        and chol < 200
        and heart_rate < 110
        and diabetes == "No"
        and bmi < 25
        and (
            "mild fatigue" in symptoms
            or "occasional tiredness" in symptoms
            or "mild dizziness" in symptoms
            or "occasional headache" in symptoms
        )
    ):

        return (

            "Low Cardiovascular Risk",

            "Low confidence",

            "Symptoms do not strongly indicate major cardiovascular abnormalities."
        )

    # =====================================================
    # HEART ATTACK
    # =====================================================

    elif (

        "left arm pain" in symptoms
        or "sweating" in symptoms
        or "severe chest pain" in symptoms
    ):

        return (

            "Heart Attack",

            "High confidence",

            "Symptoms indicate possible cardiac emergency."
        )

    # =====================================================
    # ARRHYTHMIA
    # =====================================================

    elif (

        "irregular heartbeat" in symptoms
        or "palpitations" in symptoms
        or heart_rate > 170
    ):

        return (

            "Arrhythmia",

            "High confidence",

            "Irregular heart rhythm symptoms detected."
        )

    # =====================================================
    # HEART FAILURE
    # =====================================================

    elif (

        "swelling" in symptoms
        or "persistent cough" in symptoms
        or "shortness of breath" in symptoms
    ):

        return (

            "Heart Failure",

            "High confidence",

            "Symptoms indicate reduced cardiac pumping efficiency."
        )

    # =====================================================
    # HYPERTENSION
    # =====================================================

    elif bp > 160:

        return (

            "Hypertension",

            "High confidence",

            "High blood pressure values detected."
        )

    # =====================================================
    # CORONARY ARTERY DISEASE
    # =====================================================

    elif ((

        chol > 240
        or diabetes == "Yes"
        or bmi > 30
    )
    and (

        "chest pain" in symptoms
        or "chest discomfort" in symptoms
        or "shortness of breath" in symptoms
        or "fatigue" in symptoms
    )
    ):

        return (

            "Coronary Artery Disease",

            "Medium confidence",

            "Lifestyle and cholesterol levels indicate cardiovascular risk."
        )

    # =====================================================
    # ANGINA
    # =====================================================

    elif (

        "chest tightness" in symptoms
        or "mild chest pain" in symptoms
    ):

        return (

            "Angina",

            "Medium confidence",

            "Exercise-related chest discomfort detected."
        )

    # =====================================================
    # DEFAULT
    # =====================================================

    return (

        "Hypertension",

        "Medium confidence",

        "Mild cardiovascular symptoms detected."
    )

# =========================================================
# STEP 9 — CREATE DATASET
# =========================================================

rows = []

TOTAL_ROWS = 3000

for i in range(TOTAL_ROWS):

    heart_row = heart_df.sample(1).iloc[0]

    symptom_row = symptom_df.sample(1).iloc[0]

    lifestyle_row = lifestyle_df.sample(1).iloc[0]

    # =====================================================
    # HEART FEATURES
    # =====================================================

    age = int(heart_row.get("age", 50))

    sex = heart_row.get("sex", 1)

    bp = float(heart_row.get("trestbps", 120))

    chol = float(heart_row.get("chol", 200))

    heart_rate = float(heart_row.get("thalach", 150))

    cp = heart_row.get("cp", 0)

    gender = "Male" if sex == 1 else "Female"

    cp_map = {

        0: "severe chest pain",

        1: "mild chest pain",

        2: "burning chest pain",

        3: "chest discomfort"
    }

    chest_pain = cp_map.get(cp, "chest pain")

    # =====================================================
    # SYMPTOMS
    # =====================================================

    symptoms = symptom_row.get("Symptoms", "chest pain")

    # =====================================================
    # FILTER ONLY CARDIAC SYMPTOMS
    # =====================================================

    filtered_symptoms = []

    for symptom in symptoms.split(","):

        symptom = symptom.strip().lower()

        for keyword in cardiac_keywords:

            if keyword in symptom:

                filtered_symptoms.append(symptom)

    if len(filtered_symptoms) == 0:

        filtered_symptoms = ["mild fatigue"]

    final_symptoms = ", ".join(filtered_symptoms)

    final_symptoms += f", {chest_pain}"

    # =====================================================
    # LIFESTYLE FEATURES
    # =====================================================

    smoking = lifestyle_row.get("SmokerStatus", "No")

    diabetes = lifestyle_row.get("HadDiabetes", "No")

    bmi = float(lifestyle_row.get("BMI", 25))

    alcohol = lifestyle_row.get("AlcoholDrinkers", "No")

    physical_health = lifestyle_row.get("PhysicalHealthDays", 0)

    mental_health = lifestyle_row.get("MentalHealthDays", 0)

    # =====================================================
    # DETECT PRIMARY DISEASE
    # =====================================================

    disease1, confidence1, reason = detect_disease(

        final_symptoms,
        bp,
        chol,
        heart_rate,
        diabetes,
        bmi
    )

    # =====================================================
    # DETECT SECONDARY DISEASE
    # =====================================================

    disease2 = detect_secondary_disease(disease1)

    # =====================================================
    # INPUT TEXT
    # =====================================================

    input_text = f"""
symptoms: {final_symptoms}
age: {age}
gender: {gender}
blood pressure: {bp}
cholesterol: {chol}
heart rate: {heart_rate}
smoking: {smoking}
diabetes: {diabetes}
BMI: {bmi}
alcohol: {alcohol}
physical health days: {physical_health}
mental health days: {mental_health}
"""

    # =====================================================
    # HIGH RISK CHECK
    # =====================================================

    high_risk = (

        bp > 160
        or chol > 280
        or heart_rate > 170
        or diabetes == "Yes"
    )

    # =====================================================
    # OUTPUT TEXT
    # =====================================================

    if high_risk:

        output_text = f"""
Possible conditions:
1. {disease1} ({confidence1})
2. {disease2} (Medium confidence)

Reason:
{reason}

Tests:
{disease_info[disease1]['tests']}

Recommendation:
{disease_info[disease1]['recommendation']}

Precautions:
{disease_info[disease1]['precautions']}

Disclaimer:
This is not a medical diagnosis. Please consult a qualified healthcare professional.
"""

    else:

        output_text = f"""
Possible condition:
{disease1} ({confidence1})

Reason:
{reason}

Tests:
{disease_info[disease1]['tests']}

Recommendation:
{disease_info[disease1]['recommendation']}

Precautions:
{disease_info[disease1]['precautions']}

Disclaimer:
This is not a medical diagnosis. Please consult a qualified healthcare professional.
"""

    rows.append({

        "input": input_text.strip(),

        "output": output_text.strip()
    })

# =========================================================
# STEP 10 — CREATE DATAFRAME
# =========================================================

final_df = pd.DataFrame(rows)

# =========================================================
# STEP 11 — SHUFFLE
# =========================================================

final_df = final_df.sample(frac=1).reset_index(drop=True)

# =========================================================
# STEP 12 — SAVE
# =========================================================

final_df.to_csv(

    "improved_final_medical_dataset.csv",

    index=False
)

print("✅ Improved Dataset Created Successfully")

# =========================================================
# STEP 13 — DOWNLOAD
# =========================================================

from google.colab import files

files.download("improved_final_medical_dataset.csv")

✅ Improved Dataset Created Successfully


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================================================
# AI Clinical Decision Support Assistant
# COMPLETE TRAINING CODE (GOOGLE COLAB)
# =========================================================

# =========================================================
# STEP 1 — INSTALL LIBRARIES
# =========================================================

!pip install -U transformers accelerate datasets sentencepiece -q

# =========================================================
# STEP 2 — IMPORT LIBRARIES
# =========================================================

import pandas as pd

from datasets import Dataset

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)

# =========================================================
# STEP 3 — LOAD FINAL DATASET
# =========================================================

# Upload:
# final_medical_dataset.csv

from google.colab import files

uploaded = files.upload()

# =========================================================
# STEP 4 — READ DATASET
# =========================================================

df = pd.read_csv("final_medical_dataset.csv")

print("✅ Dataset Loaded")

print(df.head())

# =========================================================
# STEP 5 — LOAD PRETRAINED T5 MODEL
# =========================================================

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(model_name)

print("✅ T5 Model Loaded")

# =========================================================
# STEP 6 — CONVERT TO HUGGINGFACE DATASET
# =========================================================

dataset = Dataset.from_pandas(df)

print("✅ Converted to HuggingFace Dataset")

# =========================================================
# STEP 7 — PREPROCESS FUNCTION
# =========================================================

MAX_INPUT = 128
MAX_OUTPUT = 256

def preprocess(example):

    input_text = "medical report generation: " + example["input"]

    target_text = example["output"]

    # INPUT TOKENIZATION
    model_inputs = tokenizer(

        input_text,

        max_length=MAX_INPUT,

        truncation=True,

        padding="max_length"
    )

    # OUTPUT TOKENIZATION
    labels = tokenizer(

        target_text,

        max_length=MAX_OUTPUT,

        truncation=True,

        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

print("✅ Preprocess Function Ready")

# =========================================================
# STEP 8 — TOKENIZE DATASET
# =========================================================

tokenized_dataset = dataset.map(preprocess)

print("✅ Dataset Tokenized")

# =========================================================
# STEP 9 — TRAIN / TEST SPLIT
# =========================================================

split_dataset = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]

test_dataset = split_dataset["test"]

print("✅ Dataset Split Completed")

print("Train Size:", len(train_dataset))

print("Test Size:", len(test_dataset))

# =========================================================
# STEP 10 — TRAINING ARGUMENTS
# =========================================================

training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    learning_rate=3e-4,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=10,

    save_total_limit=2,

    report_to="none"
)

print("✅ TrainingArguments Created")

# =========================================================
# STEP 11 — CREATE TRAINER
# =========================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

print("✅ Trainer Created")

# =========================================================
# STEP 12 — START TRAINING
# =========================================================

trainer.train()

print("✅ Training Completed")

# =========================================================
# STEP 13 — SAVE MODEL
# =========================================================

model.save_pretrained("medical_model")

tokenizer.save_pretrained("medical_model")

print("✅ Model Saved")

# =========================================================
# STEP 14 — PROMPT ENGINEERING
# =========================================================

def build_prompt(user_input):

    prompt = f"""
You are a medical assistant.

Based on patient information:
{user_input}

Generate:
1. Possible conditions
2. Confidence levels
3. Reason
4. Tests
5. Recommendation
6. Precautions
7. Disclaimer

Do NOT provide final diagnosis.
"""

    return prompt

# =========================================================
# STEP 15 — GENERATE REPORT
# =========================================================

def generate_report(user_input):

    prompt = build_prompt(user_input)

    inputs = tokenizer(

        prompt,

        return_tensors="pt",

        truncation=True,

        max_length=128
    )

    outputs = model.generate(

        **inputs,

        max_length=256,

        temperature=0.3,

        top_p=0.9,

        do_sample=True
    )

    response = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True
    )

    return response

print("✅ Prediction Function Ready")

# =========================================================
# STEP 16 — TEST MODEL
# =========================================================

sample_input = """
symptoms: chest pain, fatigue
age: 55
gender: Male
blood pressure: 145
cholesterol: 250
heart rate: 160
smoking: Yes
diabetes: Yes
BMI: 31
"""

result = generate_report(sample_input)

print("\n================================")
print("HEALTHCARE REPORT")
print("================================\n")

print(result)

Saving final_medical_dataset.csv to final_medical_dataset (1).csv
✅ Dataset Loaded
                                               input  \
0  symptoms: weight loss, insomnia, diarrhea, mil...   
1  symptoms: sweating, dizziness, runny nose, wei...   
2  symptoms: tremors, rash, appetite loss, blurre...   
3  symptoms: weight gain, anxiety, runny nose, ra...   
4  symptoms: chest pain, anxiety, sore throat, we...   

                                              output  
0  Possible conditions:\n1. Arrhythmia (High conf...  
1  Possible condition:\nArrhythmia (High confiden...  
2  Possible conditions:\n1. Coronary Artery Disea...  
3  Possible conditions:\n1. Arrhythmia (High conf...  
4  Possible conditions:\n1. Heart Failure (High c...  


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

✅ T5 Model Loaded
✅ Converted to HuggingFace Dataset
✅ Preprocess Function Ready


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Dataset Tokenized
✅ Dataset Split Completed
Train Size: 400
Test Size: 100
✅ TrainingArguments Created
✅ Trainer Created


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training Completed


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model Saved
✅ Prediction Function Ready

HEALTHCARE REPORT

Possible conditions: 1. Heart Failure (High confidence) 2. Cardiomyopathy (Medium confidence) Reason: Symptoms and clinical features indicate possible cardiovascular conditions. Tests: MRI, Echocardiogram Recommendation: Follow regular cardiac checkups Precautions: Limit strenuous exercise Disclaimer: This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
def generate_report(user_input):

    prompt = build_prompt(user_input)

    inputs = tokenizer(

        prompt,

        return_tensors="pt",

        truncation=True,

        max_length=128
    )

    outputs = model.generate(

        **inputs,

        max_length=256,

        temperature=0.3,

        top_p=0.9,

        do_sample=True
    )

    response = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True
    )

    # FORMAT OUTPUT
    response = response.replace("Possible conditions:", "\nPossible conditions:\n")

    response = response.replace("Reason:", "\nReason:\n")

    response = response.replace("Tests:", "\nTests:\n")

    response = response.replace("Recommendation:", "\nRecommendation:\n")

    response = response.replace("Precautions:", "\nPrecautions:\n")

    response = response.replace("Disclaimer:", "\nDisclaimer:\n")

    return response

In [ ]:
sample_input = """
symptoms: severe chest pain, sweating, shortness of breath, left arm pain
age: 67
gender: Male
blood pressure: 170
cholesterol: 310
heart rate: 180
smoking: Yes
diabetes: Yes
BMI: 34
alcohol: Yes
physical health days: 15
mental health days: 4
"""

print(generate_report(sample_input))


Possible conditions:
 1. Arrhythmia (High confidence) 2. Heart Failure (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Holter Monitor 
Recommendation:
 Consult heart specialist 
Precautions:
 Avoid caffeine and stress 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: chest discomfort, fatigue, breathlessness while walking
age: 59
gender: Female
blood pressure: 150
cholesterol: 260
heart rate: 145
smoking: No
diabetes: Yes
BMI: 31
alcohol: No
physical health days: 8
mental health days: 3
"""

print(generate_report(sample_input))


Possible conditions:
 1. Atrial Fibrillation (High confidence) 2. Hypertension (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Holter Monitor 
Recommendation:
 Consult heart specialist 
Precautions:
 Avoid caffeine and stress 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: irregular heartbeat, dizziness, fainting
age: 48
gender: Male
blood pressure: 135
cholesterol: 190
heart rate: 190
smoking: No
diabetes: No
BMI: 25
alcohol: Yes
physical health days: 4
mental health days: 6
"""

print(generate_report(sample_input))


Possible conditions:
 1. Heart Failure (High confidence) 2. Atrial Fibrillation (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Troponin Test 
Recommendation:
 Seek emergency medical attention immediately 
Precautions:
 Avoid physical exertion 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: swelling in legs, fatigue, persistent cough, shortness of breath
age: 72
gender: Female
blood pressure: 160
cholesterol: 240
heart rate: 155
smoking: Yes
diabetes: Yes
BMI: 36
alcohol: No
physical health days: 20
mental health days: 5
"""

print(generate_report(sample_input))


Possible conditions:
 1. Hypertension (High confidence) 2. Hypertension (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Troponin Test 
Recommendation:
 Seek emergency medical attention immediately 
Precautions:
 Avoid physical exertion 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: chest tightness during exercise, mild chest pain
age: 54
gender: Male
blood pressure: 145
cholesterol: 230
heart rate: 150
smoking: Yes
diabetes: No
BMI: 29
alcohol: Yes
physical health days: 6
mental health days: 2
"""

print(generate_report(sample_input))


Possible conditions:
 1. Atrial Fibrillation (High confidence) 2. Hypertension (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Holter Monitor 
Recommendation:
 Consult heart specialist 
Precautions:
 Avoid caffeine and stress 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: headache, dizziness, fatigue
age: 50
gender: Female
blood pressure: 180
cholesterol: 210
heart rate: 130
smoking: No
diabetes: Yes
BMI: 32
alcohol: No
physical health days: 7
mental health days: 4
"""

print(generate_report(sample_input))


Possible conditions:
 1. Hypertension (High confidence) 2. Heart Failure (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Troponin Test 
Recommendation:
 Seek emergency medical attention immediately 
Precautions:
 Avoid physical exertion 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: rapid heartbeat, chest fluttering, weakness
age: 63
gender: Male
blood pressure: 148
cholesterol: 225
heart rate: 185
smoking: Yes
diabetes: No
BMI: 30
alcohol: Yes
physical health days: 9
mental health days: 5
"""

print(generate_report(sample_input))


Possible conditions:
 1. Heart Failure (High confidence) 2. Atrial Fibrillation (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Troponin Test 
Recommendation:
 Seek emergency medical attention immediately 
Precautions:
 Avoid physical exertion 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: fatigue, swollen legs, dizziness, shortness of breath
age: 61
gender: Female
blood pressure: 155
cholesterol: 250
heart rate: 165
smoking: No
diabetes: Yes
BMI: 35
alcohol: No
physical health days: 14
mental health days: 6
"""

print(generate_report(sample_input))


Possible conditions:
 1. Hypertension (High confidence) 2. Hypertension (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Troponin Test 
Recommendation:
 Seek emergency medical attention immediately 
Precautions:
 Avoid physical exertion 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: fatigue, chest discomfort, mild dizziness
age: 57
gender: Male
blood pressure: 145
cholesterol: 235
heart rate: 148
smoking: Yes
diabetes: Yes
BMI: 30
alcohol: No
physical health days: 10
mental health days: 3
"""

print(generate_report(sample_input))


Possible conditions:
 1. Atrial Fibrillation (High confidence) 2. Hypertension (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Holter Monitor 
Recommendation:
 Consult heart specialist 
Precautions:
 Avoid caffeine and stress 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: mild fatigue, occasional dizziness
age: 32
gender: Female
blood pressure: 118
cholesterol: 170
heart rate: 92
smoking: No
diabetes: No
BMI: 22
alcohol: No
physical health days: 1
mental health days: 1
"""

print(generate_report(sample_input))


Possible conditions:
 1. Heart Failure (High confidence) 2. Hypertension (Medium confidence) 
Reason:
 Symptoms and clinical features indicate possible cardiovascular conditions. 
Tests:
 ECG, Troponin Test 
Recommendation:
 Seek emergency medical attention immediately 
Precautions:
 Avoid physical exertion 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
# =========================================================
# IMPROVED DATASET PREPARATION
# AI Clinical Decision Support Assistant
# =========================================================

# =========================================================
# STEP 1 — IMPORT LIBRARIES
# =========================================================

import pandas as pd
import random

# =========================================================
# STEP 2 — LOAD DATASETS
# =========================================================

heart_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/heart.csv"
)

symptom_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/Healthcare.csv"
)

lifestyle_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/heart_2022_no_nans.csv"
)

knowledge_df = pd.read_excel(
    "/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/disease_knowledge.xlsx"
)

# =========================================================
# STEP 3 — CLEAN COLUMN NAMES
# =========================================================

heart_df.columns = heart_df.columns.str.strip()
symptom_df.columns = symptom_df.columns.str.strip()
lifestyle_df.columns = lifestyle_df.columns.str.strip()
knowledge_df.columns = knowledge_df.columns.str.strip()

# =========================================================
# STEP 4 — CREATE DISEASE KNOWLEDGE DICTIONARY
# =========================================================

disease_info = {}

for _, row in knowledge_df.iterrows():

    disease_info[row["disease"]] = {

        "tests": row["tests"],

        "recommendation": row["recommendation"],

        "precautions": row["precautions"]
    }

# =========================================================
# STEP 5 — RULE-BASED DISEASE DETECTION
# =========================================================

def detect_disease(symptoms, bp, chol, heart_rate, diabetes, bmi):

    symptoms = symptoms.lower()

    # =====================================================
    # HEART ATTACK
    # =====================================================

    if (
        "left arm pain" in symptoms
        or "sweating" in symptoms
        or "severe chest pain" in symptoms
    ):

        return (
            "Heart Attack",
            "High confidence",
            "Symptoms indicate possible cardiac emergency."
        )

    # =====================================================
    # ARRHYTHMIA
    # =====================================================

    elif (
        "irregular heartbeat" in symptoms
        or "palpitations" in symptoms
        or heart_rate > 170
    ):

        return (
            "Arrhythmia",
            "High confidence",
            "Irregular heart rhythm symptoms detected."
        )

    # =====================================================
    # HEART FAILURE
    # =====================================================

    elif (
        "swelling" in symptoms
        or "persistent cough" in symptoms
        or "shortness of breath" in symptoms
    ):

        return (
            "Heart Failure",
            "High confidence",
            "Symptoms indicate reduced cardiac pumping efficiency."
        )

    # =====================================================
    # HYPERTENSION
    # =====================================================

    elif bp > 160:

        return (
            "Hypertension",
            "High confidence",
            "High blood pressure values detected."
        )

    # =====================================================
    # CORONARY ARTERY DISEASE
    # =====================================================

    elif (
        chol > 240
        or diabetes == "Yes"
        or bmi > 30
    ):

        return (
            "Coronary Artery Disease",
            "Medium confidence",
            "Lifestyle and cholesterol levels indicate cardiovascular risk."
        )

    # =====================================================
    # ANGINA
    # =====================================================

    elif (
        "chest tightness" in symptoms
        or "mild chest pain" in symptoms
    ):

        return (
            "Angina",
            "Medium confidence",
            "Exercise-related chest discomfort detected."
        )

    # =====================================================
    # DEFAULT
    # =====================================================

    return (
        "Cardiomyopathy",
        "Medium confidence",
        "Symptoms indicate possible cardiovascular abnormality."
    )

# =========================================================
# STEP 6 — CREATE FINAL DATASET
# =========================================================

rows = []

TOTAL_ROWS = 3000

for i in range(TOTAL_ROWS):

    # =====================================================
    # RANDOM ROWS
    # =====================================================

    heart_row = heart_df.sample(1).iloc[0]

    symptom_row = symptom_df.sample(1).iloc[0]

    lifestyle_row = lifestyle_df.sample(1).iloc[0]

    # =====================================================
    # HEART DATASET FEATURES
    # =====================================================

    age = int(heart_row.get("age", 50))

    sex = heart_row.get("sex", 1)

    bp = float(heart_row.get("trestbps", 120))

    chol = float(heart_row.get("chol", 200))

    heart_rate = float(heart_row.get("thalach", 150))

    cp = heart_row.get("cp", 0)

    # =====================================================
    # GENDER
    # =====================================================

    gender = "Male" if sex == 1 else "Female"

    # =====================================================
    # CHEST PAIN MAPPING
    # =====================================================

    cp_map = {
        0: "severe chest pain",
        1: "mild chest pain",
        2: "burning chest pain",
        3: "chest discomfort"
    }

    chest_pain = cp_map.get(cp, "chest pain")

    # =====================================================
    # SYMPTOM DATASET FEATURES
    # =====================================================

    symptoms = symptom_row.get("Symptoms", "chest pain")

    # =====================================================
    # LIFESTYLE DATASET FEATURES
    # =====================================================

    smoking = lifestyle_row.get("SmokerStatus", "No")

    diabetes = lifestyle_row.get("HadDiabetes", "No")

    bmi = float(lifestyle_row.get("BMI", 25))

    alcohol = lifestyle_row.get("AlcoholDrinkers", "No")

    physical_health = lifestyle_row.get("PhysicalHealthDays", 0)

    mental_health = lifestyle_row.get("MentalHealthDays", 0)

    # =====================================================
    # CREATE FINAL SYMPTOMS
    # =====================================================

    final_symptoms = f"{symptoms}, {chest_pain}"

    # =====================================================
    # DETECT PRIMARY DISEASE
    # =====================================================

    disease1, confidence1, reason = detect_disease(
        final_symptoms,
        bp,
        chol,
        heart_rate,
        diabetes,
        bmi
    )

    # =====================================================
    # SECONDARY DISEASES
    # =====================================================

    secondary_options = [
        "Hypertension",
        "Coronary Artery Disease",
        "Angina",
        "Atrial Fibrillation"
    ]

    disease2 = random.choice(secondary_options)

    if disease2 == disease1:
        disease2 = "Coronary Artery Disease"

    # =====================================================
    # CREATE INPUT
    # =====================================================

    input_text = f"""
symptoms: {final_symptoms}
age: {age}
gender: {gender}
blood pressure: {bp}
cholesterol: {chol}
heart rate: {heart_rate}
smoking: {smoking}
diabetes: {diabetes}
BMI: {bmi}
alcohol: {alcohol}
physical health days: {physical_health}
mental health days: {mental_health}
"""

    # =====================================================
    # HIGH RISK DETECTION
    # =====================================================

    high_risk = (
        bp > 160
        or chol > 280
        or heart_rate > 170
        or diabetes == "Yes"
    )

    # =====================================================
    # CREATE OUTPUT
    # =====================================================

    if high_risk:

        output_text = f"""
Possible conditions:
1. {disease1} ({confidence1})
2. {disease2} (Medium confidence)

Reason:
{reason}

Tests:
{disease_info[disease1]['tests']}

Recommendation:
{disease_info[disease1]['recommendation']}

Precautions:
{disease_info[disease1]['precautions']}

Disclaimer:
This is not a medical diagnosis. Please consult a qualified healthcare professional.
"""

    else:

        output_text = f"""
Possible condition:
{disease1} ({confidence1})

Reason:
{reason}

Tests:
{disease_info[disease1]['tests']}

Recommendation:
{disease_info[disease1]['recommendation']}

Precautions:
{disease_info[disease1]['precautions']}

Disclaimer:
This is not a medical diagnosis. Please consult a qualified healthcare professional.
"""

    # =====================================================
    # APPEND ROW
    # =====================================================

    rows.append({

        "input": input_text.strip(),

        "output": output_text.strip()
    })

# =========================================================
# STEP 7 — CREATE FINAL DATAFRAME
# =========================================================

final_df = pd.DataFrame(rows)

# =========================================================
# STEP 8 — SHUFFLE DATASET
# =========================================================

final_df = final_df.sample(frac=1).reset_index(drop=True)

# =========================================================
# STEP 9 — DISPLAY SAMPLE
# =========================================================

print("\n================================")
print("FINAL DATASET SAMPLE")
print("================================\n")

print(final_df.head())

# =========================================================
# STEP 10 — SAVE DATASET
# =========================================================

final_df.to_csv(
    "improved_final_medical_dataset.csv",
    index=False
)

print("\n✅ Improved Dataset Created Successfully")

# =========================================================
# STEP 11 — DOWNLOAD DATASET
# =========================================================

from google.colab import files

files.download("improved_final_medical_dataset.csv")


FINAL DATASET SAMPLE

                                               input  \
0  symptoms: sore throat, headache, appetite loss...   
1  symptoms: anxiety, back pain, headache, joint ...   
2  symptoms: abdominal pain, rash, nausea, severe...   
3  symptoms: sore throat, muscle pain, headache, ...   
4  symptoms: joint pain, back pain, diarrhea, blu...   

                                              output  
0  Possible conditions:\n1. Arrhythmia (High conf...  
1  Possible conditions:\n1. Heart Attack (High co...  
2  Possible conditions:\n1. Heart Attack (High co...  
3  Possible conditions:\n1. Arrhythmia (High conf...  
4  Possible condition:\nHeart Attack (High confid...  

✅ Improved Dataset Created Successfully


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================================================
# TRAINING CODE
# =========================================================

!pip install -U transformers accelerate datasets sentencepiece -q

import pandas as pd

from datasets import Dataset

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/promptEngineeringHealthCare/output/improved_final_medical_dataset (1).csv")

# =========================================================
# LOAD MODEL
# =========================================================

model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(model_name)

# =========================================================
# PROMPT ENGINEERING
# =========================================================

def build_prompt(user_input):

    return f"""
You are an experienced cardiology clinical assistant.

Analyze the patient symptoms carefully.

Use contextual information including:
- age
- gender
- blood pressure
- cholesterol
- heart rate
- smoking
- diabetes
- BMI

Predict ONLY medically relevant cardiovascular conditions.

Generate:
1. Possible conditions
2. Confidence levels
3. Reasoning
4. Recommended tests
5. Recommendations
6. Precautions
7. Disclaimer

Do NOT provide final diagnosis.

Patient Information:
{user_input}
"""

# =========================================================
# CONVERT DATASET
# =========================================================

dataset = Dataset.from_pandas(df)

# =========================================================
# PREPROCESS
# =========================================================

MAX_INPUT = 256
MAX_OUTPUT = 256

def preprocess(example):

    input_text = build_prompt(example["input"])

    target_text = example["output"]

    model_inputs = tokenizer(

        input_text,

        max_length=MAX_INPUT,

        truncation=True,

        padding="max_length"
    )

    labels = tokenizer(

        target_text,

        max_length=MAX_OUTPUT,

        truncation=True,

        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# =========================================================
# TOKENIZE
# =========================================================

tokenized_dataset = dataset.map(preprocess)

# =========================================================
# SPLIT
# =========================================================

split_dataset = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]

test_dataset = split_dataset["test"]

# =========================================================
# TRAINING ARGUMENTS
# =========================================================

training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    learning_rate=2e-4,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=20,

    save_total_limit=2,

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

# =========================================================
# TRAIN MODEL
# =========================================================

trainer.train()

# =========================================================
# SAVE MODEL
# =========================================================

model.save_pretrained("medical_model")

tokenizer.save_pretrained("medical_model")

print("✅ Training Completed")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training Completed


In [ ]:
# =========================================================
# TESTING CODE
# =========================================================

import torch

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration
)

# =========================================================
# DEVICE
# =========================================================

device = torch.device(

    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", device)

# =========================================================
# LOAD SAVED MODEL
# =========================================================

model = T5ForConditionalGeneration.from_pretrained(
    "medical_model"
)

tokenizer = T5Tokenizer.from_pretrained(
    "medical_model"
)

model.to(device)

print("✅ Model Loaded")

# =========================================================
# PROMPT ENGINEERING
# =========================================================

def build_prompt(user_input):

    return f"""
You are an experienced cardiology clinical assistant.

Analyze the patient symptoms carefully.

Use contextual information including:
- age
- gender
- blood pressure
- cholesterol
- heart rate
- smoking
- diabetes
- BMI

Predict ONLY medically relevant cardiovascular conditions.

Generate:
1. Possible conditions
2. Confidence levels
3. Reasoning
4. Recommended tests
5. Recommendations
6. Precautions
7. Disclaimer

Do NOT provide final diagnosis.

Patient Information:
{user_input}
"""

# =========================================================
# REPORT GENERATION
# =========================================================

def generate_report(user_input):

    prompt = build_prompt(user_input)

    inputs = tokenizer(

        prompt,

        return_tensors="pt",

        truncation=True,

        max_length=256
    )

    # MOVE TO GPU
    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    outputs = model.generate(

        **inputs,

        max_length=256,

        temperature=0.2,

        top_p=0.9,

        do_sample=True
    )

    response = tokenizer.decode(

        outputs[0],

        skip_special_tokens=True
    )

    # =====================================================
    # FORMAT OUTPUT
    # =====================================================

    response = response.replace(
        "Possible conditions:",
        "\nPossible conditions:\n"
    )

    response = response.replace(
        "Possible condition:",
        "\nPossible condition:\n"
    )

    response = response.replace(
        "Reason:",
        "\nReason:\n"
    )

    response = response.replace(
        "Tests:",
        "\nTests:\n"
    )

    response = response.replace(
        "Recommendation:",
        "\nRecommendation:\n"
    )

    response = response.replace(
        "Precautions:",
        "\nPrecautions:\n"
    )

    response = response.replace(
        "Disclaimer:",
        "\nDisclaimer:\n"
    )

    return response

print("✅ Testing Function Ready")

Using Device: cuda


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Model Loaded
✅ Testing Function Ready


In [ ]:
sample_input = """
symptoms: severe chest pain, sweating, left arm pain
age: 67
gender: Male
blood pressure: 170
cholesterol: 310
heart rate: 180
smoking: Yes
diabetes: Yes
BMI: 34
"""

print(generate_report(sample_input))


Possible conditions:
 1. Heart Attack (High confidence) 2. Coronary Artery Disease (Medium confidence) 
Reason:
 Symptoms indicate possible cardiac emergency. 
Tests:
 ECG, Troponin Test 
Recommendation:
 Seek emergency medical attention immediately 
Precautions:
 Avoid physical exertion 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: mild fatigue, slight chest discomfort
age: 42
gender: Male
blood pressure: 132
cholesterol: 205
heart rate: 102
smoking: No
diabetes: No
BMI: 26
"""

print(generate_report(sample_input))


Possible condition:
 Angina (Medium confidence) 
Reason:
 Exercise-related chest discomfort detected. 
Tests:
 ECG, Stress Test 
Recommendation:
 Consult cardiologist 
Precautions:
 Avoid heavy physical activity 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: occasional headache, dizziness
age: 50
gender: Female
blood pressure: 145
cholesterol: 215
heart rate: 110
smoking: No
diabetes: Yes
BMI: 29
"""

print(generate_report(sample_input))


Possible conditions:
 1. Coronary Artery Disease (Medium confidence) 2. Angina (Medium confidence) 
Reason:
 Lifestyle and cholesterol levels indicate cardiovascular risk. 
Tests:
 Stress Test, ECG 
Recommendation:
 Follow heart-healthy lifestyle 
Precautions:
 Reduce cholesterol intake 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: occasional tiredness
age: 29
gender: Female
blood pressure: 118
cholesterol: 170
heart rate: 88
smoking: No
diabetes: No
BMI: 21
"""

print(generate_report(sample_input))


Possible condition:
 Hypertension (Medium confidence) 
Reason:
 Mild cardiovascular symptoms detected. 
Tests:
 Blood Pressure Monitoring, ECG 
Recommendation:
 Reduce sodium intake 
Precautions:
 Exercise regularly 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
sample_input = """
symptoms: mild fatigue, slight dizziness
age: 32
gender: Female
blood pressure: 122
cholesterol: 180
heart rate: 92
smoking: No
diabetes: No
BMI: 22
alcohol: No
physical health days: 1
mental health days: 1
"""

print(generate_report(sample_input))


Possible condition:
 Coronary Artery Disease (Medium confidence) 
Reason:
 Lifestyle and cholesterol levels indicate cardiovascular risk. 
Tests:
 Stress Test, ECG 
Recommendation:
 Follow heart-healthy lifestyle 
Precautions:
 Reduce cholesterol intake 
Disclaimer:
 This is not a medical diagnosis. Please consult a qualified healthcare professional.


In [ ]:
import torch

print(torch.cuda.is_available())

True


In [ ]:
!nvidia-smi

Thu May  7 01:28:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----